# 05 — Xception Standalone Model

Xception transfer learning model fine-tuned on 299×299 MRI images for 4-class brain tumor classification.  
Two-stage training: Stage 1 freezes the Xception base and trains only the classification head; Stage 2 unfreezes the last 20 base layers for fine-tuning.  
Saves `xception_model.h5` to `../saved_models/` for use in 06_Xception_Ensemble.ipynb and 08_AllModelsCombined.ipynb.

## Section 1: Imports & Configuration

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import Xception
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize
print("✓ Imports complete")

## Section 2: Constants & Hyperparameters

In [ ]:
NOTEBOOK_NAME = "05_Xception_Standalone"

DATASET_PATH     = "../MRI_DATASET/"
TRAIN_DIR        = DATASET_PATH + "Training/"
TEST_DIR         = DATASET_PATH + "Testing/"

CLASS_NAMES      = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES      = 4

# Xception requires 299×299 input
IMG_HEIGHT       = 299
IMG_WIDTH        = 299
CHANNELS         = 3

BATCH_SIZE       = 32
EPOCHS_STAGE1    = 10   # Frozen base: train top layers only
EPOCHS_STAGE2    = 10   # Unfrozen: fine-tune last 20 layers
LEARNING_RATE    = 1e-4
FINE_TUNE_LR     = 1e-5
VALIDATION_SPLIT = 0.2
RANDOM_SEED      = 42

SAVED_MODELS_DIR = "../saved_models/"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print("✓ Constants configured")
print(f"  Image size : {IMG_HEIGHT}×{IMG_WIDTH} (Xception requires 299×299)")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Stage 1 epochs: {EPOCHS_STAGE1} (frozen base)")
print(f"  Stage 2 epochs: {EPOCHS_STAGE2} (fine-tune last 20 layers)")
print(f"  Classes    : {CLASS_NAMES}")

## Section 3: Data Loading & Verification

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='training',
    seed=RANDOM_SEED,
    shuffle=True
)
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='validation',
    seed=RANDOM_SEED,
    shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print("=" * 50)
print("DATA VERIFICATION")
print(f"Class indices      : {train_generator.class_indices}")
print(f"Training samples   : {train_generator.samples}")
print(f"Validation samples : {val_generator.samples}")
print(f"Test samples       : {test_generator.samples}")
print(f"Image size         : {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"Batch size         : {BATCH_SIZE}")
print("=" * 50)

## Section 4: Data Preprocessing & Augmentation

In [ ]:
print("✓ Data augmentation configured via ImageDataGenerator")

## Section 5: Model Definition

In [ ]:
# --- Load Xception base with ImageNet weights ---
base_model = Xception(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS))
base_model.trainable = False   # Freeze all base layers during Stage 1
print(f"✓ Xception base loaded. Total layers: {len(base_model.layers)}")

# --- Add classification head ---
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu', name='feature_layer')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(NUM_CLASSES, activation='softmax', name='output_layer')(x)

model = models.Model(inputs=base_model.input, outputs=output, name='xception_model')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(f"✓ Model compiled for Stage 1 (lr={LEARNING_RATE})")
model.summary()

## Section 6: Model Training

In [ ]:
# ─── STAGE 1: Train classification head (base frozen) ───────────────────────────
print("=" * 50)
print("STAGE 1: Training top layers (base frozen)")
print("=" * 50)

callbacks_stage1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath=SAVED_MODELS_DIR + 'xception_stage1_best.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

history1 = model.fit(
    train_generator,
    epochs=EPOCHS_STAGE1,
    validation_data=val_generator,
    callbacks=callbacks_stage1,
    verbose=1
)
print("✓ Stage 1 complete")

# ─── STAGE 2: Fine-tune last 20 layers ───────────────────────────────────────
print("\n" + "=" * 50)
print("STAGE 2: Fine-tuning last 20 layers of Xception base")
print("=" * 50)

for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(f"✓ Recompiled for Stage 2 (lr={FINE_TUNE_LR})")

callbacks_stage2 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath=SAVED_MODELS_DIR + 'xception_stage2_best.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1)
]

history2 = model.fit(
    train_generator,
    epochs=EPOCHS_STAGE2,
    validation_data=val_generator,
    callbacks=callbacks_stage2,
    verbose=1
)
print("✓ Stage 2 (fine-tuning) complete")

# --- Combined history plot ---
combined_acc   = history1.history['accuracy']     + history2.history['accuracy']
combined_val   = history1.history['val_accuracy'] + history2.history['val_accuracy']
combined_loss  = history1.history['loss']         + history2.history['loss']
combined_vloss = history1.history['val_loss']     + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(combined_acc, label='Train Accuracy')
ax1.plot(combined_val, label='Val Accuracy')
ax1.axvline(x=EPOCHS_STAGE1, color='r', linestyle='--', label='Fine-tune start')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(combined_loss,  label='Train Loss')
ax2.plot(combined_vloss, label='Val Loss')
ax2.axvline(x=EPOCHS_STAGE1, color='r', linestyle='--', label='Fine-tune start')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.suptitle(f'{NOTEBOOK_NAME} — Training History (Stage 1 + Stage 2)')
plt.tight_layout()
plt.show()

## Section 7: Model Evaluation

In [ ]:
def evaluate_model(model, generator, model_name="Model"):
    """Standard evaluation: confusion matrix, classification report, ROC, PR curves."""
    generator.reset()
    y_pred_proba = model.predict(generator, verbose=1)
    y_pred       = np.argmax(y_pred_proba, axis=1)
    y_true       = generator.classes
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} — Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()
    print(f"\n{model_name} — Classification Report")
    print("=" * 60)
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3])
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} — ROC Curve')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_proba[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_pred_proba[:, i])
        plt.plot(recall, precision, label=f'{cls} (AP = {ap:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{model_name} — Precision-Recall Curve')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    return y_pred, y_pred_proba

y_pred, y_pred_proba = evaluate_model(model, test_generator, model_name=NOTEBOOK_NAME)
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"\n✓ Test Accuracy : {test_acc:.4f}")
print(f"✓ Test Loss     : {test_loss:.4f}")

## Section 8: Save Model

In [ ]:
model_path = SAVED_MODELS_DIR + 'xception_model.h5'
model.save(model_path)
print(f"✓ Model saved to: {model_path}")
print("  This model is used as:")
print("  1. Standalone Xception classifier (08_AllModelsCombined.ipynb)")
print("  2. Feature extractor backbone (06_Xception_Ensemble.ipynb)")

## Section 9: Results Summary

In [ ]:
print("=" * 60)
print(f"NOTEBOOK: {NOTEBOOK_NAME}")
print(f"Dataset  : {train_generator.samples + val_generator.samples} training images")
print(f"Classes  : {CLASS_NAMES}")
print(f"Image size: {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Stage 1: {EPOCHS_STAGE1} epochs (lr={LEARNING_RATE}, base frozen)")
print(f"Stage 2: {EPOCHS_STAGE2} epochs (lr={FINE_TUNE_LR}, last 20 layers unfrozen)")
print(f"Seed     : {RANDOM_SEED}")
print("-" * 60)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Loss     : {test_loss:.4f}")
print("=" * 60)
print("Saved models location:", SAVED_MODELS_DIR)